# Tái lập dữ liệu MassSpecGym (`msg_simulation`) cho MARASON / ICEBERG trên Kaggle

Notebook đi theo đúng trình tự của repo gốc:

1. Cài môi trường (torch 2.6 + CUDA 12.4, DGL, RDKit…)
2. Clone repo `ms-pred`
3. Tải `MassSpecGym1.5.tsv` từ Hugging Face
4. Tạo `spec_files.hdf5` (repo gốc không có script cho bước này)
5. Chạy MAGMA + subformula
6. Tạo `data/spec_datasets/msg_simulation` bằng **script chính thức** `data_scripts/create_msg_simulation_dataset.py`
7. Kiểm tra đủ file cho MARASON

**Quy trình chính thức** (giống upstream) cần thêm 2 bảng ứng viên retrieval của MassSpecGym
(`cands_df_test_formula_256.tsv`, `cands_df_test_mass_256.tsv`, tải từ link Dropbox trong README của repo).
Hãy upload chúng thành một Kaggle Dataset rồi *Add Input* vào notebook này.
Nếu không có, notebook dùng **quy trình dự phòng**: dữ liệu train giống hệt, nhưng thiếu bảng retrieval
và các file debug, nên không chạy được bước đánh giá retrieval của MARASON.

**Trước khi chạy:** Settings → *Internet: On*. Chạy lần lượt từ trên xuống, **đừng `import torch` trước ô cài đặt.**

## 0. Cấu hình

In [ ]:
import os

REPO_URL = "https://github.com/coleygroup/ms-pred.git"  # đổi sang fork của bạn nếu cần
REPO_DIR = "/kaggle/working/ms-pred"                     # ô 4 ghi converter vào đường dẫn này
TMP_DIR  = "/tmp/msg_raw"                                # file TSV gốc (không lưu vào output)
WORKERS  = os.cpu_count()

# Thư mục chứa 2 bảng ứng viên retrieval (Kaggle Dataset bạn đã Add Input), ví dụ "/kaggle/input/msg-retrieval".
# Có đủ 2 file → quy trình chính thức. None → quy trình dự phòng.
RETRIEVAL_DIR = None

# Chỉ dùng cho quy trình dự phòng, để chạy thử nhanh với ít phổ. Tái lập dữ liệu: để None (= 119,029 phổ).
N_SPECS = None

# msg_simulation/spec_files.hdf5 và magma_outputs là symlink sang msg/ (đúng như script chính thức).
# Đặt True để thay bằng bản copy thật, cần khi đóng gói msg_simulation thành Kaggle Dataset riêng (tốn gấp đôi dung lượng).
MATERIALIZE_LINKS = False
print("CPU:", WORKERS)

## 1. Cài môi trường

Mất khoảng 5–10 phút. Ô này **dừng ngay với thông báo rõ ràng** nếu một gói bắt buộc không cài được.

- **torch 2.6.0**: lấy từ PyPI (bản Linux đã kèm CUDA 12.4), không cần `download.pytorch.org`.
- **DGL**: bắt buộc, lấy wheel dựng sẵn từ `data.dgl.ai`.
- **torch-scatter**: thử wheel dựng sẵn từ `data.pyg.org`. Nếu không có bản khớp, dùng bản thay thế viết bằng
  torch thuần (chỉ gồm 5 hàm mà ms-pred dùng), nên không bao giờ phải build từ mã nguồn.
- **pygmtools**: lấy từ PyPI. Chỉ cần cho GLACIER, MARASON và ICEBERG-Score, không cần cho MAGMA hay ICEBERG-Gen.

Cảnh báo kiểu `torchtune ... requires torchdata==0.11.0` hay `torchaudio ... requires torch==...` là của các gói
Kaggle cài sẵn mà pipeline không dùng, bỏ qua được.

In [ ]:
TORCH_SCATTER_FALLBACK = '"""Pure-PyTorch fallback for the torch_scatter functions ms-pred uses.\n\nOnly installed when no prebuilt torch-scatter wheel matches the runtime.\nUntouched output positions are 0, as in torch_scatter. scatter_max/min return\n(values, None): ms-pred never reads the arg indices.\n"""\nimport torch\n\n__version__ = "0.0-fallback"\n\n\ndef _broadcast(index, src, dim):\n    dim = dim % src.dim()\n    if index.dim() == 1:\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape)\n    return index.expand_as(src), dim\n\n\ndef _reduce(src, index, dim, dim_size, reduce):\n    index, dim = _broadcast(index, src, dim)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    shape = list(src.shape)\n    shape[dim] = dim_size\n    out = torch.zeros(shape, dtype=src.dtype, device=src.device)\n    return out.scatter_reduce_(dim, index, src, reduce, include_self=False)\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    reduce = {"add": "sum", "max": "amax", "min": "amin"}.get(reduce, reduce)\n    return _reduce(src, index, dim, dim_size, reduce)\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    return _reduce(src, index, dim, dim_size, "sum")\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    return _reduce(src, index, dim, dim_size, "mean")\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    return _reduce(src, index, dim, dim_size, "amax"), None\n\n\ndef scatter_min(src, index, dim=-1, out=None, dim_size=None):\n    return _reduce(src, index, dim, dim_size, "amin"), None\n\n\ndef scatter_softmax(src, index, dim=-1, dim_size=None):\n    full_index, d = _broadcast(index, src, dim)\n    max_per = _reduce(src, index, dim, dim_size, "amax")\n    exp = (src - max_per.gather(d, full_index)).exp()\n    sum_per = _reduce(exp, index, dim, dim_size, "sum")\n    return exp / sum_per.gather(d, full_index)\n'

import socket, subprocess, sys, time, site, os

print("Python", sys.version.split()[0])
for host in ["pypi.org", "data.dgl.ai", "data.pyg.org", "github.com", "huggingface.co"]:
    try:
        socket.gethostbyname(host); print("OK  ", host)
    except OSError:
        print("LỖI ", host, "(không phân giải được tên miền — kiểm tra Settings → Internet: On)")

def pip(*args, retries=3):
    for attempt in range(1, retries + 1):
        if subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode == 0:
            return True
        print(f"  pip thất bại (lần {attempt}/{retries})"); time.sleep(5)
    return False

if not pip("torch==2.6.0", "torchvision==0.21.0"):
    raise RuntimeError("Không cài được torch 2.6.0 từ PyPI")
if not pip("dgl", "--only-binary", ":all:", "-f", "https://data.dgl.ai/wheels/torch-2.6/cu124/repo.html"):
    raise RuntimeError("Không cài được DGL (cần truy cập data.dgl.ai)")
if not pip("rdkit==2025.03.*", "pytorch-lightning", "torchdata<0.10", "h5py", "pathos", "omegaconf", "einops",
           "linsatnet", "platformdirs", "seaborn", "scikit-learn", "pyyaml", "huggingface_hub"):
    raise RuntimeError("Không cài được các gói phụ thuộc từ PyPI")
if not pip("pygmtools"):
    print("CẢNH BÁO: không cài được pygmtools — MAGMA và ICEBERG-Gen vẫn chạy được")

if not pip("torch-scatter", "--only-binary", ":all:", "-f", "https://data.pyg.org/whl/torch-2.6.0+cu124.html", retries=1):
    print("Không có wheel torch-scatter khớp → dùng bản thay thế bằng torch thuần")
    target = os.path.join(site.getsitepackages()[0], "torch_scatter")
    os.makedirs(target, exist_ok=True)
    with open(os.path.join(target, "__init__.py"), "w") as f:
        f.write(TORCH_SCATTER_FALLBACK)

# Kiểm tra trong một tiến trình Python mới, tránh dính bản torch cũ đã nạp trong kernel
subprocess.run([sys.executable, "-c",
    "import torch, dgl, torch_scatter, rdkit; "
    "print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU', torch.cuda.is_available()); "
    "print('dgl', dgl.__version__, '| torch_scatter', torch_scatter.__version__, '| rdkit', rdkit.__version__)"],
    check=True)

## 2. Clone repo

In [ ]:
import sys, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.environ["PYTHONPATH"] = os.path.join(REPO_DIR, "src") + ":" + os.environ.get("PYTHONPATH", "")
print(os.getcwd())

In [ ]:
# Kiểm tra môi trường
import torch, dgl, torch_scatter, rdkit
import ms_pred.common as common
assert torch.__version__.startswith("2.6"), f"torch {torch.__version__}: hãy Restart kernel rồi chạy lại từ ô 0"
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| dgl", dgl.__version__, "| rdkit", rdkit.__version__)

## 3. Tải MassSpecGym1.5.tsv

In [ ]:
from huggingface_hub import hf_hub_download
TSV_PATH = hf_hub_download(
    repo_id="roman-bushuiev/MassSpecGym",
    filename="data/MassSpecGym1.5.tsv",
    repo_type="dataset",
    local_dir=TMP_DIR,
)
print(TSV_PATH, round(os.path.getsize(TSV_PATH) / 1e6), "MB")

## 4. Chọn quy trình và chuẩn bị converter

In [ ]:
import json, shutil, subprocess
from pathlib import Path
import pandas as pd

MSG_DIR = Path("data/spec_datasets/msg")
SIM_DIR = Path("data/spec_datasets/msg_simulation")
CANDS = ["cands_df_test_formula_256.tsv", "cands_df_test_mass_256.tsv"]

def run(cmd):
    # Chạy lệnh shell, in output ngay trong notebook, dừng nếu lỗi
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (mã {p.returncode}): {cmd}")

OFFICIAL = RETRIEVAL_DIR is not None and all((Path(RETRIEVAL_DIR) / c).exists() for c in CANDS)
if RETRIEVAL_DIR is not None and not OFFICIAL:
    print("Không thấy đủ bảng retrieval trong", RETRIEVAL_DIR, "→", [c for c in CANDS if not (Path(RETRIEVAL_DIR) / c).exists()])
if OFFICIAL and N_SPECS is not None:
    raise ValueError("Quy trình chính thức luôn dùng toàn bộ dữ liệu: đặt N_SPECS = None")
print("Quy trình:", "CHÍNH THỨC (giống upstream)" if OFFICIAL else "DỰ PHÒNG")
print("msg/labels.tsv của repo:", sum(1 for _ in open(MSG_DIR / "labels.tsv")) - 1, "phổ")

Ô dưới ghi script `create_msg_spec_files.py` vào repo (repo gốc chưa có script này).

In [ ]:
%%writefile /kaggle/working/ms-pred/data_scripts/create_msg_spec_files.py
#!/usr/bin/env python3
"""Convert MassSpecGym1.5.tsv into the repository's spec_files.hdf5 layout.

Each spectrum is stored as one SIRIUS-style ``.ms`` string under the key
``{identifier}.ms``, which is what ``common.HDF5Dataset.read_str`` and
``common.parse_spectra`` expect (see run_magma.py and
data_scripts/forms/01_assign_subformulae.py). Example entry:

    >compound MassSpecGymID0000001
    >formula C16H17NO4
    >parentmass 288.1225
    >ionization [M+H]+
    >instrumentation Orbitrap
    >smiles CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC
    >InChIKey VFMQMACUYWGDOJ

    >collision 30
    91.0542 0.2316
    ...

Rows without a known collision energy are written under ``>collision nan``
(the key ICEBERG uses for missing energies) unless ``--missing-ce skip`` is set.
"""

from __future__ import annotations

import argparse
import json
import os
from pathlib import Path

import h5py
import pandas as pd


DEFAULT_SOURCE_TABLE = (
    "https://huggingface.co/datasets/roman-bushuiev/MassSpecGym/"
    "resolve/main/data/MassSpecGym1.5.tsv"
)
DEFAULT_OUTPUT = Path("data/spec_datasets/msg/spec_files.hdf5")

META_COLUMNS = {
    "formula": "formula",
    "parentmass": "precursor_mz",
    "ionization": "adduct",
    "instrumentation": "instrument_type",
    "smiles": "smiles",
    "InChIKey": "inchikey",
}


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source-table", default=DEFAULT_SOURCE_TABLE,
                        help="Path or URL of MassSpecGym1.5.tsv")
    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument("--labels", type=Path, default=None,
                        help="Optional labels.tsv; only specs in its 'spec' column are written")
    parser.add_argument("--missing-ce", choices=["nan", "skip"], default="nan",
                        help="How to handle rows without a collision energy")
    parser.add_argument("--id-col", default="identifier")
    parser.add_argument("--mz-col", default="mzs")
    parser.add_argument("--inten-col", default="intensities")
    parser.add_argument("--ce-col", default="collision_energy")
    parser.add_argument("--chunksize", type=int, default=20000)
    parser.add_argument("--limit", type=int, default=None,
                        help="Stop after writing this many spectra (for quick checks)")
    parser.add_argument("--overwrite", action="store_true")
    parser.add_argument("--audit-out", type=Path, default=None)
    return parser.parse_args()


def format_collision_energy(value) -> str | None:
    """Return the energy label used in '>collision {label}', or None if unknown."""
    try:
        val = float(value)
    except (TypeError, ValueError):
        return None
    if pd.isna(val):
        return None
    return str(int(val)) if val.is_integer() else f"{val:g}"


def split_peaks(value) -> list[str]:
    if not isinstance(value, str):
        return []
    return [tok.strip() for tok in value.split(",") if tok.strip()]


def build_ms_string(row: pd.Series, args, ce_label: str) -> str | None:
    """Build one SIRIUS .ms string, or None if the peak lists are unusable."""
    mzs = split_peaks(row[args.mz_col])
    intens = split_peaks(row[args.inten_col])
    if len(mzs) == 0 or len(mzs) != len(intens):
        return None
    try:
        peaks = [(float(m), float(i)) for m, i in zip(mzs, intens)]
    except ValueError:
        return None

    header = [f">compound {row[args.id_col]}"]
    for key, col in META_COLUMNS.items():
        val = row.get(col)
        if val is not None and not pd.isna(val) and str(val).strip():
            header.append(f">{key} {str(val).strip()}")

    # The blank line separates metadata from the spectrum block; parse_spectra
    # relies on it to tell the two groups apart.
    peak_lines = [f"{m} {i}" for m, i in peaks]
    return "\n".join(header) + f"\n\n>collision {ce_label}\n" + "\n".join(peak_lines) + "\n"


def write_str(h5: h5py.File, name: str, data: str):
    """Same on-disk layout as common.HDF5Dataset.write_str."""
    dt = h5py.special_dtype(vlen=str)
    ds = h5.create_dataset(name, (1,), dtype=dt, compression="gzip")
    ds[0] = data


def main():
    args = get_args()

    if args.output.exists() and not args.overwrite:
        raise FileExistsError(f"{args.output} exists; pass --overwrite to replace it")

    keep_specs = None
    if args.labels is not None:
        keep_specs = set(pd.read_csv(args.labels, sep="\t", usecols=["spec"])["spec"].astype(str))

    required = {args.id_col, args.mz_col, args.inten_col, args.ce_col}
    audit = {
        "source_table": str(args.source_table),
        "source_rows": 0,
        "written": 0,
        "written_missing_ce": 0,
        "skipped_not_in_labels": 0,
        "skipped_missing_ce": 0,
        "skipped_bad_peaks": 0,
        "skipped_duplicate_id": 0,
    }
    seen = set()

    args.output.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = args.output.with_name(args.output.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    done = False
    with h5py.File(tmp_path, "w") as h5:
        for chunk in pd.read_csv(args.source_table, sep="\t", chunksize=args.chunksize):
            missing = required - set(chunk.columns)
            if missing:
                raise KeyError(f"Missing columns {sorted(missing)}; available: {list(chunk.columns)}")

            for _, row in chunk.iterrows():
                audit["source_rows"] += 1
                spec = str(row[args.id_col])

                if keep_specs is not None and spec not in keep_specs:
                    audit["skipped_not_in_labels"] += 1
                    continue
                if spec in seen:
                    audit["skipped_duplicate_id"] += 1
                    continue

                ce_label = format_collision_energy(row[args.ce_col])
                if ce_label is None:
                    if args.missing_ce == "skip":
                        audit["skipped_missing_ce"] += 1
                        continue
                    ce_label = "nan"

                ms_str = build_ms_string(row, args, ce_label)
                if ms_str is None:
                    audit["skipped_bad_peaks"] += 1
                    continue

                write_str(h5, f"{spec}.ms", ms_str)
                seen.add(spec)
                audit["written"] += 1
                if ce_label == "nan":
                    audit["written_missing_ce"] += 1

                if args.limit is not None and audit["written"] >= args.limit:
                    done = True
                    break
            if done:
                break

    os.replace(tmp_path, args.output)

    if keep_specs is not None:
        audit["labels_specs"] = len(keep_specs)
        audit["labels_specs_missing_from_output"] = len(keep_specs - seen)

    print(json.dumps(audit, indent=2))
    if args.audit_out is not None:
        args.audit_out.parent.mkdir(parents=True, exist_ok=True)
        args.audit_out.write_text(json.dumps(audit, indent=2))


if __name__ == "__main__":
    main()

## 5A. Quy trình chính thức

Giống upstream: dựng `msg/` trước (`spec_files.hdf5`, `retrieval/`, `subformulae/`, `magma_outputs/`),
rồi để script chính thức sinh `msg_simulation/`. `msg/labels.tsv` có sẵn trong repo (119,029 phổ),
nên MAGMA chạy trên đúng tập phổ của simulation challenge.

In [ ]:
if OFFICIAL:
    (MSG_DIR / "retrieval").mkdir(parents=True, exist_ok=True)
    for c in CANDS:
        shutil.copy2(Path(RETRIEVAL_DIR) / c, MSG_DIR / "retrieval" / c)
    run(f"python data_scripts/create_msg_spec_files.py --source-table {TSV_PATH} "
        f"--labels {MSG_DIR}/labels.tsv --output {MSG_DIR}/spec_files.hdf5 "
        f"--audit-out {MSG_DIR}/spec_files_audit.json --overwrite")
else:
    print("Bỏ qua (quy trình dự phòng)")

MAGMA + subformula trên `msg/`. **Bước lâu nhất**, có thể mất nhiều giờ với 4 CPU.

In [ ]:
if OFFICIAL:
    import time; t0 = time.time()
    run(f"workers={WORKERS} bash data_scripts/dag/run_magma.sh msg")
    print(f"MAGMA + subformula: {(time.time() - t0) / 60:.1f} phút")
else:
    print("Bỏ qua (quy trình dự phòng)")

In [ ]:
if OFFICIAL:
    run(f"python data_scripts/create_msg_simulation_dataset.py --source-table {TSV_PATH} "
        f"--source-dataset {MSG_DIR} --output-dataset {SIM_DIR} --force-output --overwrite "
        f"--audit-out {SIM_DIR}/audit.json")
    audit = json.loads((SIM_DIR / "audit.json").read_text())
    print("Số phổ:", audit["output_rows"], "| imputed:", audit["imputed_check"])
else:
    print("Bỏ qua (quy trình dự phòng)")

## 5B. Quy trình dự phòng (không có bảng retrieval)

Tạo `labels.tsv` và `splits/split.tsv` theo đúng quy tắc và định dạng của script chính thức
(cùng cột, cùng thứ tự dòng, có cột `collision_imputed`), rồi tạo `spec_files.hdf5` và chạy MAGMA ngay trong `msg_simulation/`.

In [ ]:
if not OFFICIAL:
    SIM_DIR.joinpath("splits").mkdir(parents=True, exist_ok=True)
    cols = ["identifier", "smiles", "inchikey", "formula", "precursor_mz", "adduct",
            "instrument_type", "collision_energy", "fold", "simulation_challenge"]
    df = pd.read_csv(TSV_PATH, sep="\t", usecols=cols)
    print("Tổng số dòng:", len(df))
    df = df[df["simulation_challenge"].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})].copy()

    def fmt_ce(v):
        try:
            v = float(v)
        except (TypeError, ValueError):
            return None
        if pd.isna(v):
            return None
        return f"['{int(v)}']" if v.is_integer() else f"['{v:g}']"

    df["collision_energies"] = df["collision_energy"].map(fmt_ce)
    df = df.dropna(subset=["collision_energies"]).drop_duplicates(subset=["identifier"], keep="first")
    fold = df["fold"].astype(str).str.strip().str.lower()
    df["split"] = fold.map(lambda f: "train" if f == "train" else ("val" if f in {"val", "valid", "validation"} else "test"))
    print("Simulation challenge + có năng lượng:", len(df))   # kỳ vọng 119,029

    if N_SPECS is not None and N_SPECS < len(df):
        df = df.groupby("split", group_keys=False).sample(frac=N_SPECS / len(df), random_state=0).sort_index()
        print("CHẠY THỬ với", len(df), "phổ — không phải dữ liệu tái lập đầy đủ")

    labels = pd.DataFrame({
        "dataset": "MassSpecGym",
        "spec": df["identifier"].astype(str),
        "ionization": df["adduct"],
        "formula": df["formula"],
        "smiles": df["smiles"],
        "inchikey": df["inchikey"],
        "instrument": df["instrument_type"],
        "collision_energies": df["collision_energies"],
        "precursor": df["precursor_mz"],
        "collision_imputed": False,
    })
    labels.to_csv(SIM_DIR / "labels.tsv", sep="\t", index=False)
    pd.DataFrame({"name": labels["spec"], "split": df["split"].values}).to_csv(
        SIM_DIR / "splits" / "split.tsv", sep="\t", index=False)
    print(df["split"].value_counts().to_dict())
else:
    print("Bỏ qua (quy trình chính thức)")

In [ ]:
if not OFFICIAL:
    run(f"python data_scripts/create_msg_spec_files.py --source-table {TSV_PATH} "
        f"--labels {SIM_DIR}/labels.tsv --output {SIM_DIR}/spec_files.hdf5 "
        f"--audit-out {SIM_DIR}/spec_files_audit.json --overwrite")
    import time; t0 = time.time()
    run(f"workers={WORKERS} bash data_scripts/dag/run_magma.sh msg_simulation")
    print(f"MAGMA + subformula: {(time.time() - t0) / 60:.1f} phút")
else:
    print("Bỏ qua (quy trình chính thức)")

## 6. Kiểm tra dữ liệu cho MARASON

In [ ]:
import ms_pred.common as common

if MATERIALIZE_LINKS:
    for name in ["spec_files.hdf5", "magma_outputs"]:
        p = SIM_DIR / name
        if p.is_symlink():
            target = p.resolve(); p.unlink()
            (shutil.copytree if target.is_dir() else shutil.copy2)(target, p)
            print("Đã thay symlink bằng bản copy:", p)

need = ["labels.tsv", "splits/split.tsv", "spec_files.hdf5",
        "magma_outputs/magma_tree.hdf5", "subformulae/no_subform.hdf5"]
if OFFICIAL:
    need += ["retrieval/" + c for c in CANDS] + ["labels_debug.tsv", "splits/split_debug.tsv", "spec_files_debug.hdf5"]
ok = True
for rel in need:
    p = SIM_DIR / rel
    exists = p.exists()
    ok &= exists
    link = f" → {os.readlink(p)}" if p.is_symlink() else ""
    size = (sum(f.stat().st_size for f in p.rglob("*")) if p.is_dir() else p.stat().st_size) / 1e6 if exists else 0
    print(f"{'OK   ' if exists else 'THIẾU'} {rel:45s} {size:9.1f} MB{link}")

labels = pd.read_csv(SIM_DIR / "labels.tsv", sep="\t")
print("\nSố phổ trong labels.tsv:", len(labels), "(tái lập đầy đủ = 119,029)")
h5 = common.HDF5Dataset(SIM_DIR / "spec_files.hdf5")
meta, specs = common.parse_spectra(h5.read_str(f"{labels.spec.iloc[0]}.ms").split("\n"))
print("Phổ mẫu:", meta.get("compound"), "| CE:", list(specs.keys()), "| số peak:", specs.num_peaks)
h5.close()
print("\nSẴN SÀNG cho MARASON" if ok else "\nCÒN THIẾU FILE — xem các dòng THIẾU ở trên")

## 7. Bước tiếp theo: chạy MARASON gốc

Dữ liệu trong `data/spec_datasets/msg_simulation` giờ là đầu vào cho `run_scripts/marason/`. Các bước còn lại
là **train**, không phải chuẩn bị dữ liệu (theo `run_scripts/marason/run_all.sh`, phần MassSpecGym):

1. `python launcher_scripts/run_from_config.py configs/marason/marason_train_msg.yaml`: train bộ sinh mảnh
2. `python launcher_scripts/run_from_config.py configs/marason/marason_gen_predict_train_msg.yaml`: dự đoán DAG cho tập train
3. `python data_scripts/dag/add_dag_intens.py --pred-dag-path results/marason_msg_simulation/split_rnd1/preds_train_100/tree_preds.hdf5 --true-dag-path data/spec_datasets/msg_simulation/subformulae/no_subform.hdf5 --out-dag-path results/marason_msg_simulation/split_rnd1/preds_train_100_inten.hdf5 --num-workers 4 --add-raw`
4. **Phổ tham chiếu (RAG):** `configs/marason/marason_inten_train_msg.yaml` đọc sẵn từ
   `data/msg_simulation/closest_neighbors/infinite` (`load-reference: true`). Lần đầu phải tạo thư mục đó
   (`mkdir -p`) và chạy với `load-reference: [false]`, `save-reference: [true]` để tính và lưu.
5. `python launcher_scripts/run_from_config.py configs/marason/marason_inten_train_msg.yaml`: train MARASON

Config gốc dùng 200 epoch và GPU lớn (`visible_devices: [0, 1]`, `num-workers: 16`); trên Kaggle cần giảm
`visible_devices`, `num-workers`, `batch-size` và thường không train hết được trong một phiên 12 giờ.

## Lưu kết quả

Bấm **Save Version** để Kaggle giữ `/kaggle/working/ms-pred/data/spec_datasets` (cả `msg/` và `msg_simulation/`).
Biến output đó thành Kaggle Dataset để dùng lại mà không phải chạy MAGMA lần nữa.